# Análise Exploratória — Predição da Alfabetização Infantil

**Tech Challenge Fase 3 — PosTech FIAP | IA Science**

Este notebook documenta a análise exploratória da camada Gold (`gold.ml_features_alunos_v2`, construída sobre a pipeline da Fase 2) e as **hipóteses analíticas** que orientaram as decisões de modelagem.

Os gráficos aqui gerados são os mesmos produzidos por `src/visualization/eda_plots.py` (executável de forma reproduzível fora do notebook).

In [ ]:
import os
import sys

sys.path.insert(0, "..")
os.environ.setdefault("GOOGLE_APPLICATION_CREDENTIALS", "../credentials/service-account.json")

import google.auth
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from google.cloud import bigquery

import config

sns.set_theme(style="whitegrid")
credentials, _ = google.auth.default()
client = bigquery.Client(project=config.GCP_PROJECT_ID, credentials=credentials)
print("Conectado ao projeto:", config.GCP_PROJECT_ID)

## 1. Carregamento da camada Gold

A base de modelagem tem grão de **aluno** (`id_aluno`), enriquecida com variáveis territoriais, socioeconômicas e de metas agregadas por município.

In [ ]:
from src.visualization.eda_plots import load_gold_data

df = load_gold_data(client)
print(f"Registros: {len(df):,}")
df.head()

In [ ]:
df.describe().T

## 2. Distribuição da variável alvo

`alfabetizado` é derivado do corte oficial de **743 pontos na escala SAEB** (Pesquisa Alfabetiza Brasil, 2023).

> **Decisão de modelagem #1:** a variável `proficiencia` (nota bruta) foi **excluída das features**, pois `alfabetizado = 'Sim'` equivale exatamente a `proficiencia >= 743` — usá-la seria vazamento direto do alvo.

In [ ]:
dist = df["alfabetizado"].value_counts(normalize=True) * 100
print(dist)

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="alfabetizado", hue="alfabetizado", palette="Blues_r", legend=False)
plt.title("Distribuição da variável alvo")
plt.show()

O desbalanceamento é moderado (~59% / 41%).

> **Decisão de modelagem #2:** tratamos via `class_weight="balanced"` / `scale_pos_weight` em vez de undersampling — não há motivo para descartar ~600 mil linhas da classe majoritária.

## 3. Valores ausentes

Nem todo NULL é problema: alguns são **estruturais** e informativos.

In [ ]:
nulos = pd.DataFrame({
    "nulos": df.isna().sum(),
    "pct": (df.isna().mean() * 100).round(2),
})
nulos[nulos["nulos"] > 0].sort_values("pct", ascending=False)

`taxa_alfabetizacao_escola_prior` é nula para todas as linhas de **2023** — não existe edição 2022 na base para servir de histórico.

> **Decisão de modelagem #3:** em vez de fabricar um valor, mantivemos o NULL (imputado pela mediana dentro do pipeline) e adicionamos a flag `tem_historico_escola`, para o modelo distinguir "escola sem histórico" de "escola com histórico ruim". Mesmo princípio da política de NULL documentada na Fase 2.

## 4. Correlações

Quais variáveis se associam mais ao desfecho?

In [ ]:
from src.visualization.eda_plots import NUMERIC_FEATURES

df_corr = df[NUMERIC_FEATURES].copy()
df_corr["target"] = df["alfabetizado"].map({"Não": 0, "Nao": 0, "Sim": 1})

corr = df_corr.corr()
plt.figure(figsize=(11, 9))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="Blues", square=True)
plt.title("Matriz de correlação")
plt.show()

corr["target"].drop("target").sort_values(ascending=False)

## 5. INSE e desempenho

**Hipótese H1:** municípios com maior nível socioeconômico (INSE) apresentam maior taxa de alfabetização.

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x="alfabetizado", y="inse_municipio", hue="alfabetizado",
            palette="Set2", legend=False)
plt.title("INSE municipal por status de alfabetização")
plt.show()

## 6. Rede de ensino

**Hipótese H2:** há diferença de desempenho entre redes Municipal e Estadual.

In [ ]:
taxa_rede = df.groupby("rede")["alfabetizado"].apply(
    lambda s: (s == "Sim").mean() * 100).sort_values(ascending=False)
print(taxa_rede)

taxa_rede.plot(kind="bar", figsize=(6, 4), color="#2980b9")
plt.ylabel("% alfabetizados")
plt.title("Taxa de alfabetização por rede")
plt.xticks(rotation=0)
plt.show()

## 7. Disparidade territorial

**Hipótese H3:** existe forte disparidade regional — a UF do aluno carrega sinal preditivo próprio, além do município.

In [ ]:
UF_NOMES = {
    "11": "RO", "12": "AC", "13": "AM", "14": "RR", "15": "PA", "16": "AP", "17": "TO",
    "21": "MA", "22": "PI", "23": "CE", "24": "RN", "25": "PB", "26": "PE", "27": "AL",
    "28": "SE", "29": "BA", "31": "MG", "32": "ES", "33": "RJ", "35": "SP", "41": "PR",
    "42": "SC", "43": "RS", "50": "MS", "51": "MT", "52": "GO", "53": "DF",
}

df["uf"] = df["sigla_uf_code"].map(UF_NOMES)
taxa_uf = df.groupby("uf")["alfabetizado"].apply(
    lambda s: (s == "Sim").mean() * 100).sort_values(ascending=False)

plt.figure(figsize=(12, 5))
taxa_uf.plot(kind="bar", color="#16a085")
plt.ylabel("% alfabetizados")
plt.title("Taxa de alfabetização por UF")
plt.axhline(taxa_uf.mean(), color="red", linestyle="--", label="Média nacional")
plt.legend()
plt.show()

print(f"Amplitude entre UFs: {taxa_uf.max() - taxa_uf.min():.1f} pontos percentuais")

## 8. Metas de alfabetização

**Hipótese H4:** a distância entre a taxa atual e a meta 2030 (`gap_meta_2030`) sinaliza municípios estruturalmente frágeis.

In [ ]:
plt.figure(figsize=(9, 5))
sns.histplot(data=df, x="gap_meta_2030", hue="alfabetizado", bins=50,
             element="step", stat="density", common_norm=False)
plt.axvline(0, color="red", linestyle="--", label="Meta atingida")
plt.title("Distância até a meta 2030 por status de alfabetização")
plt.legend()
plt.show()

## 9. Duplicidade de alunos entre edições

Verificação que motivou a decisão mais importante do projeto.

In [ ]:
q = f"""
WITH counts AS (
  SELECT id_aluno, COUNT(*) AS n
  FROM `{config.GCP_PROJECT_ID}.{config.BQ_DATASET_GOLD}.{config.ML_FEATURES_TABLE}`
  WHERE rede IN ('Municipal', 'Estadual')
  GROUP BY id_aluno
)
SELECT COUNT(*) AS alunos,
       SUM(CASE WHEN n > 1 THEN 1 ELSE 0 END) AS duplicados,
       ROUND(SAFE_DIVIDE(SUM(CASE WHEN n > 1 THEN 1 ELSE 0 END), COUNT(*)) * 100, 2) AS pct
FROM counts
"""
client.query(q).to_dataframe()

> **Decisão de modelagem #4 (a mais importante):** ~51% dos alunos aparecem em duas linhas (edições 2023 e 2024). Um `train_test_split` aleatório por linha colocaria o **mesmo aluno** em treino e teste, inflando artificialmente as métricas.
>
> Por isso todo o projeto usa `GroupShuffleSplit` / `StratifiedGroupKFold` **agrupados por `id_aluno`**.

## 10. Síntese das hipóteses e decisões

| # | Hipótese | Veredito | Consequência na modelagem |
|---|---|---|---|
| H1 | INSE ↑ → alfabetização ↑ | Confirmada, efeito moderado | `inse_municipio` mantida |
| H2 | Rede diferencia desempenho | Confirmada, efeito pequeno | `rede` mantida (one-hot) |
| H3 | Disparidade territorial forte | Confirmada, amplitude alta entre UFs | `sigla_uf_code` adicionada |
| H4 | Gap até a meta sinaliza fragilidade | Confirmada | Metas incorporadas como features |

**Decisões de leakage derivadas da EDA:**
1. `proficiencia` excluída (define o alvo);
2. split agrupado por `id_aluno` (51% de duplicidade);
3. histórico de escola apenas do **ano anterior**;
4. agregados municipais casados pelo **mesmo ano** da linha;
5. `TargetEncoder` em `id_escola` testado e **descartado** — capturava o resultado dos colegas da mesma prova (detalhes no README).